# Image Tuning with DimRed API

This notebook demonstrates how to run prompt tuning with image data using the DimRed API.

## Overview

The workflow:
1. Load image dataset from `~/data/image_example.json`
2. Create a project and dataset
3. Add datapoints with image payloads
4. Create a prompt for image classification
5. Create a metric
6. Run tuning
7. Poll for results

## Setup

In [ ]:
import json
import logging
import sys
import os
from pathlib import Path
from dimred_api_client import DimRedAPIClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger(__name__)

## Configuration

In [ ]:
# API Configuration
API_KEY = os.environ.get("DIMRED_API_KEY")
BASE_URL = "https://api.dimred.com"

# Path to image dataset
IMAGE_DATASET_PATH = os.path.join(os.path.dirname(os.path.abspath('')), '..', 'data', 'image_example.json')

# Initialize client
client = DimRedAPIClient(API_KEY, BASE_URL)
print(f"✓ Initialized DimRed API client")
print(f"✓ Dataset path: {IMAGE_DATASET_PATH}")

## Step 1: Load Image Dataset

The dataset contains datapoints with image payloads. Each datapoint has:
- `input_data`: The question/instruction
- `expected_output`: The expected response
- `payloads`: Array of payloads, including images with base64-encoded data

In [ ]:
# Load the image dataset
with open(IMAGE_DATASET_PATH, 'r') as f:
    image_dataset = json.load(f)

print(f"Loaded {len(image_dataset)} datapoints from {IMAGE_DATASET_PATH}")
print(f"\nFirst datapoint structure:")
print(f"  - input_data keys: {list(image_dataset[0]['input_data'].keys())}")
print(f"  - expected_output keys: {list(image_dataset[0]['expected_output'].keys())}")
print(f"  - Number of payloads: {len(image_dataset[0]['payloads'])}")
print(f"  - Payload type: {image_dataset[0]['payloads'][0]['payload_type']}")

## Step 2: Create Project

In [ ]:
project_id = client.create_project(
    project_name="Image Classification Tuning",
    project_description="Testing image-based prompt tuning with cat detection"
)

print(f"✓ Created project: {project_id}")

## Step 3: Create Dataset

In [ ]:
dataset_id = client.create_dataset(
    project_id=project_id,
    dataset_name="Cat Detection Image Dataset"
)

print(f"✓ Created dataset: {dataset_id}")

## Step 4: Add Datapoints with Image Payloads

When adding datapoints with images, the API expects:
- `input_data`: JSON string of the input
- `expected_output`: JSON string of the expected output
- `payloads`: Array of payload objects with structure:
  ```json
  {
    "payload_type": "image",
    "payload": {
      "media_type": "image/png",
      "data": "base64_encoded_image_data"
    }
  }
  ```

In [ ]:
# Convert dataset to API format
datapoints = []
for item in image_dataset:
    datapoint = {
        "input_data": json.dumps(item["input_data"]),
        "expected_output": json.dumps(item["expected_output"]),
        "payloads": item["payloads"]  # Pass payloads as-is
    }
    datapoints.append(datapoint)

# Add to dataset
count = client.add_datapoints(dataset_id, datapoints)
print(f"✓ Added {count} datapoints with image payloads")

# Display an example datapoint structure
print("\n=== Example Datapoint Structure ===")
example = datapoints[0]
print(f"\ninput_data: {example['input_data']}")
print(f"\nexpected_output: {example['expected_output']}")
print(f"\npayloads: {len(example['payloads'])} payload(s)")

# Render the image
from IPython.display import Image, display
import base64

payload = example['payloads'][0]
print(f"\n--- Rendering Image Payload ---")
print(f"Media type: {payload['payload']['media_type']}")
print(f"Payload type: {payload['payload_type']}\n")

# Decode and display the image
image_data = base64.b64decode(payload['payload']['data'])
display(Image(data=image_data))


## Step 5: Create Prompt

Create a prompt for image-based cat detection. The prompt will receive both the text instruction and the image.

In [ ]:
messages = [
    {
        "prompt_text": (
            "You are an expert at analyzing images and identifying animals. "
            "Your task is to look at an image and determine if it shows a cat.\n\n"
            "Respond with JSON containing:\n"
            "- is_cat: true or false\n"
            "- reasoning: brief explanation of your decision based on visual features"
        ),
        "prompt_message_type": "system"
    }
]

# Output schema for structured response
output_schema = {
    "type": "object",
    "properties": {
        "is_cat": {
            "type": "boolean",
            "description": "Whether the image shows a cat"
        },
        "reasoning": {
            "type": "string",
            "description": "Brief explanation based on visual features"
        }
    },
    "required": ["is_cat", "reasoning"]
}

prompt_id = client.create_prompt(
    project_id=project_id,
    messages=messages,
    name="Cat Detection from Images",
    output_schema=output_schema
)

print(f"✓ Created prompt: {prompt_id}")

## Step 6: Create Metric

In [ ]:
metric_code = '''
import json

def metric_func(output, expected):
    """
    Check if the model correctly identified whether the image shows a cat.
    Returns 1.0 for correct classification, 0.0 for incorrect.
    """
    # Parse output and expected if they're strings
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return 0.0

    if isinstance(expected, str):
        try:
            expected = json.loads(expected)
        except json.JSONDecodeError:
            return 0.0

    # Extract is_cat field
    output_value = output.get("is_cat")
    expected_value = expected.get("is_cat")

    # Both must be present and match
    if output_value is None or expected_value is None:
        return 0.0

    # Return 1.0 if they match, 0.0 if they don't
    return 1.0 if output_value == expected_value else 0.0
'''

metric_id = client.create_metric(
    project_id=project_id,
    code=metric_code,
    metric_name="Cat Classification Accuracy",
    metric_description="Measures whether the model correctly identifies cats in images"
)

print(f"✓ Created metric: {metric_id}")

## Step 7: Run Tuning

Start the prompt tuning session. The model will receive both text instructions and images.

In [ ]:
tuning_result = client.run_tuning(
    project_id=project_id,
    dataset_id=dataset_id,
    prompt_id=prompt_id,
    metric_id=metric_id,
    num_iterations=3,
    model_name="gpt-4o",  # Use vision-capable model
    provider="openai"
)

session_id = tuning_result["tuning_session_id"]
task_id = tuning_result["task_id"]

print(f"✓ Tuning started")
print(f"  Session ID: {session_id}")
print(f"  Task ID: {task_id}")

## Step 8: Wait for Completion

Poll the tuning session until it completes. This can take 10-30 minutes.

In [ ]:
# Optional: Enable DEBUG logging to see polling progress
# logging.getLogger().setLevel(logging.DEBUG)

final_result = client.wait_for_tuning_completion(
    session_id=session_id,
    poll_interval=15,
    timeout=3600
)

print("\n=== Final Results ===")
print(f"Session ID: {final_result['session_id']}")
print(f"Status: {final_result['status']}")
print(f"Best Metric Value: {final_result.get('best_metric_value', 'N/A')}")
print(f"Iterations: {final_result.get('max_iterations', 'N/A')}")

if final_result.get("metrics"):
    print("\nMetrics:")
    print(json.dumps(final_result["metrics"], indent=2))

## Step 9: Fetch Best Prompt

In [ ]:
best_prompt = client.get_best_prompt(session_id)

print("\n=== Best Prompt ===")
print(f"Prompt ID: {best_prompt['prompt_id']}")
print(f"Prompt Name: {best_prompt.get('name', 'N/A')}")
print(f"\nMessages:")
for i, msg in enumerate(best_prompt.get('messages', []), 1):
    print(f"\n--- Message {i} ({msg.get('prompt_message_type', 'unknown')}) ---")
    print(msg.get('prompt_text', ''))

if best_prompt.get('output_schema'):
    print(f"\nOutput Schema:")
    print(json.dumps(best_prompt['output_schema'], indent=2))

## Summary

You've successfully completed image-based prompt tuning with the DimRed API:

- ✓ Loaded image dataset with base64-encoded images
- ✓ Created project and dataset
- ✓ Added datapoints with image payloads
- ✓ Created vision prompt
- ✓ Created classification metric
- ✓ Ran tuning with vision-capable model (GPT-4o)
- ✓ Retrieved optimized prompt

## Key Points for Image Payloads

1. **Image Payload Structure**: Images are passed as payloads with `payload_type: "image"` and contain `media_type` and base64 `data`
2. **Model Selection**: Use vision-capable models like `gpt-4o`, `gpt-4-vision-preview`, or `claude-3-opus`
3. **Payload Processing**: The system automatically converts image payloads to the correct format for the LLM provider
4. **Multiple Payloads**: You can have multiple payloads per datapoint (e.g., multiple images or mix of text/images)